# Part 2 — Train ML models

**Inputs from Part 1:** `src/transform.prepare_train_val_test()`  
**Handoff to Part 3:** trained models in `artifacts/models/`

Train-row features are unchanged vs the old split-then-transform approach (test is after train), so the existing saved XGBoost model remains valid. Preprocessing (`StandardScaler` / `OneHotEncoder`) stays inside the train pipeline only.

Previous: [`01_eda_and_rules.ipynb`](01_eda_and_rules.ipynb) · Next: [`03_optimize_thresholds.ipynb`](03_optimize_thresholds.ipynb)


In [ ]:
from pathlib import Path
import sys

# Project root (works when cwd is notebooks/ or project root)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    DATA_DIR,
    MODELS_DIR,
    ARTIFACTS_DIR,
    STRATEGY_DIR,
    TRAIN_CSV,
    TEST_CSV,
    XGB_MODEL_PATH,
    LOGISTIC_MODEL_PATH,
    FROZEN_STRATEGY_PATH,
)

print("Project root:", ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)


## Section 2 - Training ML models, producing fraud-probabilities:


In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler , PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
import joblib

print("Libraries Loaded")


Libraries Loaded


In [ ]:
from src.transform import prepare_train_val_test, prepare_model_frame, make_xy

# Full-timeline features, then chronological split (no model preprocessing fit here)
df_train, df_val, df_test = prepare_train_val_test()
print(df_train.shape, df_val.shape, df_test.shape)


In [ ]:
# Features already built by prepare_train_val_test()
# Existing model artifacts can be reused; retrain only if you change train features/labels.
print("Columns:", len(df_train.columns))
df_train.head()


In [80]:
df_train.columns


Index(['Unnamed: 0', 'transaction_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'running_total_amt',
       'running_avg_amt', 'running_total_transactions',
       'time_since_last_transaction', 'transactions_last_30d',
       'amt_to_avg_ratio', 'day_of_week', 'hour_of_day',
       'minutes_since_last_transaction', 'is_first_transaction', 'age',
       'cust_merch_distance_km', 'transactions_prev_10min',
       'transactions_prev_30min', 'transactions_prev_60min'],
      dtype='object')

In [81]:
df_train.head()


,Unnamed: 0,transaction_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,running_total_amt,running_avg_amt,running_total_transactions,time_since_last_transaction,transactions_last_30d,amt_to_avg_ratio,day_of_week,hour_of_day,minutes_since_last_transaction,is_first_transaction,age,cust_merch_distance_km,transactions_prev_10min,transactions_prev_30min,transactions_prev_60min
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,0.0,0.0,0,NaT,0.0,1.0,1,0,0.0,1,30,78.597568,0.0,0.0,0.0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,0.0,0.0,0,NaT,0.0,1.0,1,0,0.0,1,40,30.212176,0.0,0.0,0.0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,0.0,0.0,0,NaT,0.0,1.0,1,0,0.0,1,56,108.206083,0.0,0.0,0.0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,0.0,0.0,0,NaT,0.0,1.0,1,0,0.0,1,51,95.673231,0.0,0.0,0.0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,0.0,0.0,0,NaT,0.0,1.0,1,0,0.0,1,32,77.556744,0.0,0.0,0.0


In [82]:
cols_to_drop = ['is_fraud' , 'cc_num' , 'transaction_time' , 'time_since_last_transaction' , 'first' , 'last' , 'street' , 'zip' , 'trans_num' , 'unix_time' , 'dob' , 'total_prev_fraud' , 'prev_fraud_exists' ,'Unnamed: 0']

X_train = df_train.drop(columns = cols_to_drop)
y_train = df_train['is_fraud'].copy()

X_val = df_val.drop(columns = cols_to_drop)
y_val = df_val['is_fraud'].copy()

X_test = df_test.drop(columns = cols_to_drop)
y_test = df_test['is_fraud'].copy()


In [83]:
print(X_train.shape , y_train.shape)


(1037340, 26) (1037340,)


In [84]:
X_train.columns


Index(['merchant', 'category', 'amt', 'gender', 'city', 'state', 'lat', 'long',
       'city_pop', 'job', 'merch_lat', 'merch_long', 'running_total_amt',
       'running_avg_amt', 'running_total_transactions',
       'transactions_last_30d', 'amt_to_avg_ratio', 'day_of_week',
       'hour_of_day', 'minutes_since_last_transaction', 'is_first_transaction',
       'age', 'cust_merch_distance_km', 'transactions_prev_10min',
       'transactions_prev_30min', 'transactions_prev_60min'],
      dtype='object')

In [85]:
num_cols = ['amt' , 
           'city_pop' , 
           'running_total_amt' , 
           'running_total_transactions' , 
           'running_avg_amt' , 
            'amt_to_avg_ratio' ,
           'transactions_last_30d' , 
           'minutes_since_last_transaction' , 
           'is_first_transaction' , 
           'transactions_prev_10min' , 
           'transactions_prev_30min' , 
           'transactions_prev_60min' ,
           'cust_merch_distance_km' ,
           'merch_lat' ,
           'merch_long' ,
           'lat' ,
           'long' ,
           'age'
          ] 

cat_cols = ['merchant' ,
            'category' ,
            'gender' , 
            'city' ,
            'state' ,
            'job' ,
            'day_of_week' ,
            'hour_of_day'
           ]



In [86]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)


In [87]:
cat_pipeline = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )
])


#### Creating a function for cross-validation results summary:


In [88]:
def best_cv_result(grid_search, model_name):
    """
    
    Input: A fitted grid (grid object) , model name (str).

    Output: A dictionary that contains the model's name, and the results of the best estimator configuration.
    
    """
    
    i = grid_search.best_index_

    return {
        "model": model_name,
        "estimator": grid_search.best_estimator_,
        "params": grid_search.best_params_,
        "log_loss": -grid_search.cv_results_["mean_test_log_loss"][i],
        "brier": -grid_search.cv_results_["mean_test_brier"][i],
        "pr_auc": grid_search.cv_results_["mean_test_pr_auc"][i],
        "roc_auc": grid_search.cv_results_["mean_test_roc_auc"][i],
    }


In [89]:
best_models_results = []


#### Creating calibration-check functions:


In [111]:
def calibration_check(y_true, y_pred_prob, n_bins = 10):

    
    df = pd.DataFrame({
        "y_true": y_true,
        "y_pred_prob": y_pred_prob
    })

    df['bin'] = pd.cut(
        df['y_pred_prob'],
        bins = n_bins,
        include_lowest = True
    )

    return (
        df.groupby('bin', observed=True)
        .agg(
            mean_predicted_probability = ('y_pred_prob', 'mean'),
            observed_positive_rate = ('y_true', 'mean'),
            count = ('y_true', 'size')
        )
        .reset_index()
    )


In [122]:
def calibration_check_quantiles(y_true, y_pred_prob, n_quantiles = 10):

    
    df = pd.DataFrame({
        "y_true": y_true,
        "y_pred_prob": y_pred_prob
    })

    df['quantile'] = pd.qcut(
        df['y_pred_prob'],
        q = n_quantiles, 
        duplicates = 'drop'
    )

    return (
        df.groupby('quantile', observed = True)
        .agg(
            mean_predicted_probability=('y_pred_prob', 'mean'),
            observed_positive_rate=('y_true', 'mean'),
            count=('y_true', 'size')
        )
        .reset_index()
    )


### Logistic regression model:


In [91]:
logistic_preprocessor = ColumnTransformer([
    ("num" , StandardScaler() , num_cols) ,
    ("cat" , cat_pipeline , cat_cols) ]
)


logistic_model = LogisticRegression( 
    class_weight = None ,
    max_iter = 2000 ,
    solver = 'saga' ,
    random_state = 42
)

logistic_final_pipeline = Pipeline([
    ("preprocessor" , logistic_preprocessor) ,
    ("model" , logistic_model)
]
)


In [92]:
tscv = TimeSeriesSplit(n_splits = 3)


In [93]:
logistic_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__class_weight": [None, "balanced"]
}


In [94]:
scoring = {
    "log_loss": "neg_log_loss",
    "brier": "neg_brier_score",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc"
}


In [95]:
logistic_grid_search = GridSearchCV(
    estimator = logistic_final_pipeline,
    param_grid = logistic_param_grid,
    scoring = scoring,
    refit = "log_loss",
    cv = tscv,
    n_jobs = 2,
    verbose = 2,
    return_train_score = True
)


In [96]:
logistic_grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 6 candidates, totalling 18 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['amt',
                                                                          'city_pop',
                                                                          'running_total_amt',
                                                                          'running_total_transactions',
                                                                          'running_avg_amt',
                                                                          'amt_to_avg_ratio',
                                                                          'transactions_last_30d',
                                                                          'minutes_since_last_transaction',
                                                                          'is_fi...
                                                                          'hour_of_day'])])),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42,
                                                           solver='saga'))]),
             n_jobs=2,
             param_grid={'model__C': [0.1, 1, 10],
                         'model__class_weight': [None, 'balanced'],
                         'model__penalty': ['l2']},
             refit='log_loss', return_train_score=True,
             scoring={'brier': 'neg_brier_score', 'log_loss': 'neg_log_loss',
                      'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
             verbose=2)

In [97]:
print("Logistic Grid Search - Finished")


Logistic Grid Search - Finished


In [98]:
best_models_results.append(best_cv_result(logistic_grid_search , "Logistic Regression"))


In [99]:
joblib.dump(
    logistic_grid_search,
    MODELS_DIR / "logistic_grid_search.joblib"
)

joblib.dump(
    logistic_grid_search.best_estimator_,
    LOGISTIC_MODEL_PATH
)


['best_logistic_model.joblib']

In [100]:
print("Logistic - saved in joblib")


Logistic - saved in joblib


### XGBoost model:


In [101]:
xgb_preprocessor = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", cat_pipeline, cat_cols)
])


xgb_model = XGBClassifier(
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = 2
)


xgb_final_pipeline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", xgb_model)
])


In [102]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight_ratio = neg / pos

xgb_param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
    "model__scale_pos_weight": [
        1,
        float(np.sqrt(scale_pos_weight_ratio)),
        float(scale_pos_weight_ratio),
    ],
}

print(
    f"scale_pos_weight candidates: 1, "
    f"{np.sqrt(scale_pos_weight_ratio):.3f} (sqrt), "
    f"{scale_pos_weight_ratio:.3f} (neg/pos)"
)


In [103]:
xgb_grid_search = GridSearchCV(
    estimator = xgb_final_pipeline,
    param_grid = xgb_param_grid,
    scoring = scoring,
    refit = "log_loss",
    cv = tscv,
    n_jobs = 2,
    verbose = 2,
    return_train_score = True
)


In [104]:
xgb_grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         'passthrough',
                                                                         ['amt',
                                                                          'city_pop',
                                                                          'running_total_amt',
                                                                          'running_total_transactions',
                                                                          'running_avg_amt',
                                                                          'amt_to_avg_ratio',
                                                                          'transactions_last_30d',
                                                                          'minutes_since_last_transaction',
                                                                          'is_first...
                                                      num_parallel_tree=None, ...))]),
             n_jobs=2,
             param_grid={'model__colsample_bytree': [0.8],
                         'model__learning_rate': [0.05, 0.1],
                         'model__max_depth': [3, 5],
                         'model__n_estimators': [200, 400],
                         'model__subsample': [0.8]},
             refit='log_loss', return_train_score=True,
             scoring={'brier': 'neg_brier_score', 'log_loss': 'neg_log_loss',
                      'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
             verbose=2)

In [105]:
print("XGBoost Grid Search - Finished")


XGBoost Grid Search - Finished


In [106]:
best_models_results.append(best_cv_result(xgb_grid_search , "XGBoost"))


In [107]:
joblib.dump(
    xgb_grid_search,
    MODELS_DIR / "xgb_grid_search.joblib"
)

joblib.dump(
    xgb_grid_search.best_estimator_,
    XGB_MODEL_PATH
)


['best_xgb_model.joblib']

In [108]:
print("XGBoost - saved in joblib")


XGBoost - saved in joblib


### Checking results:


In [120]:
print("Logistic Regression - Calibration On Validation Set , By Bins")

calibration_check(
    y_val,
    logistic_grid_search.predict_proba(X_val)[:, 1]
)


Logistic Regression - Calibration On Validation Set , By Bins


,bin,mean_predicted_probability,observed_positive_rate,count
0,"(-0.002, 0.1]",0.004259,0.002764,257257
1,"(0.1, 0.2]",0.136825,0.140206,970
2,"(0.2, 0.3]",0.245985,0.293893,262
3,"(0.3, 0.4]",0.345699,0.503185,157
4,"(0.4, 0.5]",0.450158,0.597826,92
5,"(0.5, 0.6]",0.545663,0.671233,73
6,"(0.6, 0.7]",0.648715,0.681818,66
7,"(0.7, 0.8]",0.746218,0.831169,77
8,"(0.8, 0.9]",0.854322,0.746988,83
9,"(0.9, 1.0]",0.976562,0.872483,298


In [121]:
print("XGBoost - Calibration On Validation Set , By Bins")


calibration_check(
    y_val,
    xgb_grid_search.predict_proba(X_val)[:, 1]
)


XGBoost - Calibration On Validation Set , By Bins


,bin,mean_predicted_probability,observed_positive_rate,count
0,"(-0.001984, 0.1]",0.001095,0.000517,257014
1,"(0.1, 0.2]",0.140217,0.115538,502
2,"(0.2, 0.3]",0.245157,0.224409,254
3,"(0.3, 0.4]",0.348157,0.315385,130
4,"(0.4, 0.5]",0.448208,0.389474,95
5,"(0.5, 0.6]",0.546915,0.526882,93
6,"(0.6, 0.7]",0.647694,0.567568,74
7,"(0.7, 0.8]",0.751007,0.758621,87
8,"(0.8, 0.9]",0.854291,0.882353,170
9,"(0.9, 1.0]",0.969212,0.987991,916


In [123]:
print("Logistic Regression - Calibration On Validation Set , By Quantiles")

calibration_check_quantiles(
    y_val,
    logistic_grid_search.predict_proba(X_val)[:, 1]
)


Logistic Regression - Calibration On Validation Set , By Quantiles


,quantile,mean_predicted_probability,observed_positive_rate,count
0,"(-0.001, 0.000173]",0.000111,0.000617,25934
1,"(0.000173, 0.000285]",0.000228,0.000193,25933
2,"(0.000285, 0.000415]",0.000347,0.000270,25934
3,"(0.000415, 0.000597]",0.000500,0.000193,25933
4,"(0.000597, 0.000882]",0.000728,0.000424,25934
5,"(0.000882, 0.00141]",0.001119,0.000810,25933
6,"(0.00141, 0.00253]",0.001898,0.001465,25933
7,"(0.00253, 0.0051]",0.003637,0.001967,25934
8,"(0.0051, 0.0121]",0.007947,0.005707,25933
9,"(0.0121, 1.0]",0.056379,0.047659,25934


In [124]:
print("XGBoost - Calibration On Validation Set , By Quantiles")


calibration_check_quantiles(
    y_val,
    xgb_grid_search.predict_proba(X_val)[:, 1]
)


XGBoost - Calibration On Validation Set , By Quantiles


,quantile,mean_predicted_probability,observed_positive_rate,count
0,"(-0.0009844, 7.52e-05]",0.000057,0.000000,25934
1,"(7.52e-05, 0.000103]",0.000089,0.000000,25933
2,"(0.000103, 0.000134]",0.000118,0.000000,25934
3,"(0.000134, 0.000171]",0.000152,0.000000,25933
4,"(0.000171, 0.000221]",0.000195,0.000000,25934
5,"(0.000221, 0.000297]",0.000256,0.000000,25933
6,"(0.000297, 0.000433]",0.000357,0.000000,25933
7,"(0.000433, 0.000744]",0.000566,0.000077,25934
8,"(0.000744, 0.0017]",0.001109,0.000154,25933
9,"(0.0017, 1.0]",0.062618,0.059073,25934


In [125]:
best_models_results_df = pd.DataFrame(best_models_results)
best_models_results_df.sort_values('pr_auc' , ascending = False)


,model,estimator,params,log_loss,brier,pr_auc,roc_auc
1,XGBoost,"(ColumnTransformer(transformers=[('num', 'pass...","{'model__colsample_bytree': 0.8, 'model__learn...",0.007171,0.001693,0.849406,0.995043
0,Logistic Regression,"(ColumnTransformer(transformers=[('num', Stand...","{'model__C': 0.1, 'model__class_weight': None,...",0.020858,0.003973,0.359546,0.914092


In [115]:
best_xgb = xgb_grid_search.best_estimator_

feature_names = best_xgb.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb.named_steps["model"].feature_importances_

xgb_feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

xgb_feature_importance.head(20)


,feature,importance
0,num__amt,0.080667
1,cat__hour_of_day_23,0.079583
2,cat__category_grocery_pos,0.053830
3,num__amt_to_avg_ratio,0.053343
4,cat__hour_of_day_22,0.052134
5,cat__category_gas_transport,0.047964
6,cat__category_entertainment,0.043091
7,num__transactions_last_30d,0.029872
8,cat__category_grocery_net,0.023648
9,cat__category_home,0.021746


### XGBoost performed better w.r.t all metrics, and in the calibration check.

Its grid search now also tunes `scale_pos_weight` for class imbalance (`1`, `sqrt(neg/pos)`, `neg/pos`).
